#CODE

## ALS

### Note:
  - Used Implicit ALS model : A collaborative filtering model that learn from user-item ratings, just different implementations from LightFM

In [ ]:
import pandas as pd
import numpy as np
import json
import random

#### ignore this step LightFM install error

In [ ]:
#!pip install lightfm

### Implicit ALS Model

In [ ]:
# Loading csv
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/FairTrip_Project/fairtrip_philly_reviews.csv')
print(f"Shape: {df.shape}")
print(df.head())

Mounted at /content/drive
Shape: (578777, 19)
                review_id                 user_id             business_id  \
0  8JFGBuHMoiNDyfcxuWNtrA  smOvOajNG0lS4Pq7d8g4JQ  RZtGWDLCAtuipwaZ-UfjmQ   
1  Xs8Z8lmKkosqW5mw_sVAoA  IQsF3Rc6IgCzjVV9DE8KXg  eFvzHawVJofxSnD7TgbZtg   
2  J-4NdnDZ0pUQaUEEwDI9KQ  vrKkXsozqqecF3CW4cGaVQ  rjuWz_AD3WfXJc03AhIO_w   
3  JBWZmBy69VMggxj3eYn17Q  aFa96pz67TwOFu4Weq5Agg  kq5Ghhh14r-eCxlVmlyd8w   
4  YcLXh-3UC9y6YFAI9xxzPQ  G0DHgkSsDozqUPWtlxVEMw  oBhJuukGRqPVvYBfTkhuZA   

   stars  useful  funny  cool  \
0    4.0       0      0     0   
1    5.0       0      0     0   
2    5.0       2      2     2   
3    5.0       0      0     0   
4    4.0       0      0     0   

                                                text                 date  \
0  Good food--loved the gnocchi with marinara\nth...  2009-10-14 19:57:14   
1  My absolute favorite cafe in the city. Their b...  2014-11-12 15:30:27   
2  I thoroughly enjoyed the show.  Chill way to s...  2012-12

In [ ]:
!pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 19.9 MB/s eta 0:00:00


In [ ]:
import implicit
from scipy.sparse import csr_matrix

In [ ]:
# Building Sparse Matrix

# Create integer indices for users and items
df['user_idx'] = pd.Categorical(df['user_id']).codes
df['item_idx'] = pd.Categorical(df['business_id']).codes

n_users = df['user_idx'].nunique()
n_items = df['item_idx'].nunique()

print(f"Users: {n_users}, Items: {n_items}")

# Build sparse matrix (items x users) — implicit expects this orientation
sparse_item_user = csr_matrix(
    (df['stars'].astype(float), (df['item_idx'], df['user_idx'])),
    shape=(n_items, n_users)
)

Users: 37666, Items: 14344


In [ ]:
#Training Model

model = implicit.als.AlternatingLeastSquares(
    factors=64,
    iterations=30,
    regularization=0.1,
    random_state=42
)

model.fit(sparse_item_user)

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
# Generate Predicted Scores
# User x Item orientation for predictions
sparse_user_item = sparse_item_user.T.tocsr()

scores_matrix = model.user_factors @ model.item_factors.T
print(f"Scores matrix shape: {scores_matrix.shape}")  # (n_users, n_items)

Scores matrix shape: (14344, 37666)


In [ ]:
#Saving everything:

#need to transpose it again
scores_matrix = scores_matrix.T
print(f"Scores matrix shape: {scores_matrix.shape}")

np.save('scores_matrix.npy', scores_matrix)

# Save mappings: original id -> index
user_id_map = dict(enumerate(pd.Categorical(df['user_id']).categories))
item_id_map = dict(enumerate(pd.Categorical(df['business_id']).categories))

with open('user_id_map.json', 'w') as f:
    json.dump(user_id_map, f)
with open('item_id_map.json', 'w') as f:
    json.dump(item_id_map, f)

print("Done — saved scores_matrix.npy, user_id_map.json, item_id_map.json")

Scores matrix shape: (37666, 14344)
Done — saved scores_matrix.npy, user_id_map.json, item_id_map.json


In [ ]:
# Save as float32 instead of float64 — cuts size in half
scores_matrix1 = scores_matrix.astype(np.float32)
np.save('/content/drive/MyDrive/FairTrip_Project/scores_matrix.npy', scores_matrix)

### Simulating Groups

In [ ]:
with open('user_id_map.json') as f:
    user_id_map = json.load(f)  # index -> user_id
with open('item_id_map.json') as f:
    item_id_map = json.load(f)  # index -> business_id

# Reverse maps: original id -> index
user_to_idx = {v: k for k, v in user_id_map.items()}
item_to_idx = {v: k for k, v in item_id_map.items()}

In [ ]:
random.seed(42)

# Get users who appear enough times to be reliable
active_users = df['user_id'].unique().tolist()

# Sample 100 groups of 3-5 users
groups = []
for _ in range(100):
    size = random.randint(3, 5)
    group = random.sample(active_users, size)
    groups.append(group)

print(f"Simulated {len(groups)} groups")
print(f"Example group: {groups[0]}")

Simulated 100 groups
Example group: ['ltqzjnqbFWfhW6GkMfGRNA', 'VjVO1ILX6ZQFa4DOmfn7Jg', '1EN03oRpBPVcSwLgLao5FQ', 'I8iITIOjYTNmPw9P2p-yLQ', 'jGX50ZdHvhjMc19Wv7Jflw']


### ALS Baseline: pick top venues per group

In [ ]:
def als_baseline(group, itinerary_size=5):
    # Pick top venues by averaging ALS scores across group members
    group_scores = []
    for user_id in group:
        user_idx = int(user_to_idx[user_id])
        group_scores.append(scores_matrix[user_idx])  # (14344,)

    # Average scores across group members
    avg_scores = np.mean(group_scores, axis=0)

    # Pick top venues
    top_indices = np.argsort(avg_scores)[::-1][:itinerary_size]
    top_venues = [item_id_map[i] for i in top_indices]

    return top_venues, avg_scores, top_indices

### Compute NSW

In [ ]:
def compute_nsw(group, top_indices):
    # Geometric mean of each member's average score for the itinerary
    utilities = []
    for user_id in group:
        user_idx = int(user_to_idx[user_id])
        user_scores = scores_matrix[user_idx][top_indices]
        utilities.append(np.mean(user_scores))

    # Geometric mean
    utilities = np.array(utilities)
    utilities = np.clip(utilities, 1e-10, None)  # avoid log(0)
    nsw = np.exp(np.mean(np.log(utilities)))
    return nsw

### Compute AUC

In [ ]:
def compute_auc(group, top_indices, threshold=4.0):
    """
    For each user, check if recommended venues are ones they actually rated highly.
    Uses held-out star ratings as ground truth.
    """
    aucs = []
    for user_id in group:
        user_reviews = df[df['user_id'] == user_id][['business_id', 'stars']]
        if len(user_reviews) == 0:
            continue

        liked = set(user_reviews[user_reviews['stars'] >= threshold]['business_id'])
        recommended = set(item_id_map[i] for i in top_indices)

        hits = len(liked & recommended)
        auc_approx = hits / max(len(liked), 1)
        aucs.append(auc_approx)

    return np.mean(aucs) if aucs else 0.0

In [ ]:
def compute_mrr(group, top_indices, threshold=4.0):
    """
    For each user, find their first relevant venue in the itinerary.
    Relevant = starred >= threshold in their review history.
    """
    mrrs = []
    for user_id in group:
        user_reviews = df[df['user_id'] == user_id][['business_id', 'stars']]
        liked = set(user_reviews[user_reviews['stars'] >= threshold]['business_id'])

        rr = 0.0
        for rank, idx in enumerate(top_indices, start=1):
            venue = item_id_map[idx]
            if venue in liked:
                rr = 1.0 / rank
                break
        mrrs.append(rr)

    return np.mean(mrrs) if mrrs else 0.0

### Run across all groups

In [ ]:
import json

# Reload item_id_map with integer keys to fix KeyError
with open('item_id_map.json') as f:
    item_id_map_str_keys = json.load(f)
    item_id_map = {int(k): v for k, v in item_id_map_str_keys.items()}

nsw_scores = []
auc_scores = []
mrr_scores = []

for group in groups:
    top_venues, avg_scores, top_indices = als_baseline(group)
    nsw = compute_nsw(group, top_indices)
    auc = compute_auc(group, top_indices)
    mrr = compute_mrr(group, top_indices)
    nsw_scores.append(nsw)
    auc_scores.append(auc)
    mrr_scores.append(mrr)

print(f"ALS Baseline — NSW: {np.mean(nsw_scores):.4f}, AUC: {np.mean(auc_scores):.4f}, MRR: {np.mean(mrr_scores):.4f}")

ALS Baseline — NSW: 0.0995, AUC: 0.1104, MRR: 0.3557


## NSW Optimizer

In [ ]:
def nsw_optimizer(group, scores_matrix, user_to_idx, item_id_map, itinerary_size=5):
    """
    Selects an itinerary by maximizing Nash Social Welfare greedily.
    Optimizes: Maximize Sum(log(User_Utility))
    """
    n_items = scores_matrix.shape[1]
    member_indices = [int(user_to_idx[user_id]) for user_id in group]

    # Extract preference matrix for this group's members
    group_user_scores = scores_matrix[member_indices, :]

    # Shift scores to ensure strictly positive utilities so log functions work safely
    min_score = group_user_scores.min()
    if min_score <= 0:
        group_user_scores = group_user_scores - min_score + 1.0

    selected_indices = []
    remaining_indices = list(range(n_items))
    user_utilities = np.full(len(group), 1e-5)

    for _ in range(itinerary_size):
        best_nsw = -np.inf
        best_idx = None
        best_updated_utilities = None

        for idx in remaining_indices:
            potential_scores = group_user_scores[:, idx]

            if len(selected_indices) == 0:
                potential_utilities = potential_scores
            else:
                potential_utilities = (user_utilities * len(selected_indices) + potential_scores) / (len(selected_indices) + 1)

            potential_utilities = np.clip(potential_utilities, 1e-10, None)
            current_nsw = np.sum(np.log(potential_utilities))

            if current_nsw > best_nsw:
                best_nsw = current_nsw
                best_idx = idx
                best_updated_utilities = potential_utilities

        if best_idx is not None:
            selected_indices.append(best_idx)
            remaining_indices.remove(best_idx)
            user_utilities = best_updated_utilities
        else:
            break

    top_venues = [item_id_map[i] for i in selected_indices]
    return top_venues, np.array(selected_indices)

In [ ]:
# Evaluate the NSW Optimizer

nsw_opt_nsw, nsw_opt_auc, nsw_opt_mrr = [], [], []

for group in groups:
    # Run the new NSW optimization framework
    top_venues, top_indices = nsw_optimizer(group, scores_matrix, user_to_idx, item_id_map)

    # Calculate performance using your existing metric functions
    nsw_opt_nsw.append(compute_nsw(group, top_indices))
    nsw_opt_auc.append(compute_auc(group, top_indices))
    nsw_opt_mrr.append(compute_mrr(group, top_indices))

print(f"NSW Optimizer - NSW: {np.mean(nsw_opt_nsw):.4f}, AUC: {np.mean(nsw_opt_auc):.4f}, MRR: {np.mean(nsw_opt_mrr):.4f}")

NSW Optimizer — NSW: 0.1142, AUC: 0.1114, MRR: 0.3521


## LLM

##Preprocess Step to Create Dataset for LLM for All the Previous Step

In [ ]:
import json
import random
import numpy as np
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# =========================
# PREPARE DATA
# =========================

# Explicitly load scores_matrix and maps for robustness
scores_matrix = np.load('/content/drive/My Drive/FairTrip_Project/ALS_results/scores_matrix.npy') # Load the scores matrix

with open('/content/drive/My Drive/FairTrip_Project/ALS_results/user_id_map.json', 'r') as f:
    # Keys loaded from JSON are strings, convert to int
    user_id_map = {int(k): v for k, v in json.load(f).items()}
with open('/content/drive/My Drive/FairTrip_Project/ALS_results/item_id_map.json', 'r') as f:
    # Keys loaded from JSON are strings, convert to int
    item_id_map = {int(k): v for k, v in json.load(f).items()}

# reverse maps (original ID <-> index)
user_to_idx = {v: k for k, v in user_id_map.items()} # user_id (string) -> index (int)
item_to_business = {k: v for k, v in item_id_map.items()} # index (int) -> business_id (string)

# =========================
# SIMULATE GROUPS
# =========================

random.seed(42)

# Get active users from the loaded map for consistency
active_users = list(user_to_idx.keys())

groups = []

for i in range(100):
    size = random.randint(3, 5)
    group = random.sample(active_users, size)

    groups.append({
        "group_id": i,
        "members": group
    })

# =========================
# BUILD SINGLE DATASET
# =========================

TOP_K = 10

dataset = []

for group_data in groups:

    group_id = group_data["group_id"]
    members = group_data["members"]

    # get matrix indices
    member_indices = [user_to_idx[u] for u in members]

    # aggregate scores
    # scores_matrix should now be (n_users, n_items) due to explicit load
    group_scores = scores_matrix[member_indices].mean(axis=0)

    # top businesses
    top_indices = np.argsort(group_scores)[::-1][:TOP_K]

    recommendations = []

    for idx in top_indices:
        recommendations.append({
            "business_id": item_to_business[idx],
            "score": round(float(group_scores[idx]), 4)
        })

    dataset.append({
        "group_id": group_id,
        "members": members,
        "recommendations": recommendations
    })

# =========================
# SAVE SINGLE JSON DATASET
# =========================

save_path = '/content/drive/My Drive/FairTrip_Project/group_recommendation_dataset.json'

with open(save_path, 'w') as f:
    json.dump(dataset, f, indent=2)

print(f"Dataset saved to: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset saved to: /content/drive/My Drive/FairTrip_Project/group_recommendation_dataset.json


In [ ]:
# Loading csv
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
userdata.get('secret-key')

## First Zero Shot Prompt : gpt 4o mini

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import asyncio

from openai import AsyncOpenAI
from tqdm.asyncio import tqdm_asyncio
from datetime import datetime
from zoneinfo import ZoneInfo

from google.colab import drive
from google.colab import userdata

# =============================================================================
# GOOGLE DRIVE
# =============================================================================

drive.mount('/content/drive')

GoogleDrivePath = '/content/drive/My Drive/FairTrip_Project/'
input_file = GoogleDrivePath + 'group_recommendation_dataset.json'

# =============================================================================
# LOAD DATA
# =============================================================================

with open(input_file, "r") as f:
    # The 'group_recommendation_dataset.json' contains a list of group dictionaries.
    # Each dictionary has keys like "group_id", "members", "recommendations".
    # The `loaded_group_data` variable will be this list.
    loaded_group_data = json.load(f)

# The 'groups' variable for the evaluation functions should be a list of member IDs.
# We extract the 'members' list from each group dictionary in `loaded_group_data`.
groups = [group_info["members"] for group_info in loaded_group_data]

# The 'item_id_map' and 'df' (reviews DataFrame) are expected to be available
# from the global scope after running previous cells (e.g., DhwCP8YnTqWS, K5O3eXt0byqP, o6ROimPfb2az).
# Therefore, we do not need to re-assign them from the `loaded_group_data`.

print(f"Loaded {len(groups)} groups")
print(f"Loaded {len(df)} reviews") # This `df` refers to the global df from previous cells.

# =============================================================================
# OPENAI CLIENT
# =============================================================================

openai_api_key = userdata.get('secret-key')

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab secrets.")

client = AsyncOpenAI(api_key=openai_api_key)

# =============================================================================
# PROMPT BUILDER
# =============================================================================

def build_prompt(group, df, candidate_venues, itinerary_size=5):
    """
    Build a zero-shot recommendation prompt for the LLM.
    """

    member_profiles = []

    for user_id in group:

        user_reviews = (
            df[df["user_id"] == user_id][["business_id", "stars"]]
            .sort_values("stars", ascending=False)
            .head(20)
            .to_dict(orient="records")
        )

        member_profiles.append({
            "user_id": user_id,
            "review_history": user_reviews
        })

    prompt = f"""
You are a group travel recommender system.

Your goal is to recommend EXACTLY {itinerary_size} venues that maximize fairness and collective satisfaction for the entire group.

A good recommendation:
- satisfies multiple members
- avoids venues any member would strongly dislike
- balances preferences fairly

==================================================
GROUP MEMBER HISTORIES
==================================================

{json.dumps(member_profiles, indent=2)}

==================================================
CANDIDATE VENUES
==================================================

Choose ONLY from these venue IDs:

{json.dumps(candidate_venues, indent=2)}

==================================================
INSTRUCTIONS
==================================================

1. Analyze each user's preferences from highly-rated venues.
2. Infer shared interests across the group.
3. Select EXACTLY {itinerary_size} venue IDs.
4. Do NOT invent venue IDs.
5. Return ONLY valid JSON.

Required JSON format:

{{
  "recommended_venue_ids": [
    "id1",
    "id2",
    "id3",
    "id4",
    "id5"
  ],
  "reasoning": "short explanation"
}}
"""

    return prompt

# =============================================================================
# SINGLE GROUP RECOMMENDATION
# =============================================================================

async def llm_recommend(
    group,
    df,
    candidate_venues,
    itinerary_size=5,
    retries=2
):

    prompt = build_prompt(
        group,
        df,
        candidate_venues,
        itinerary_size
    )

    raw = ""

    for attempt in range(retries + 1):

        try:

            response = await client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                response_format={"type": "json_object"},
                max_completion_tokens=512,
            )

            raw = response.choices[0].message.content.strip()

            parsed = json.loads(raw)

            return parsed["recommended_venue_ids"][:itinerary_size]

        except (json.JSONDecodeError, KeyError) as e:

            print(f"[Retry {attempt+1}] JSON parse error: {e}")

            if attempt == retries:
                print(f"\n[FAILED GROUP]")
                print(group)
                print(raw)
                return []

            await asyncio.sleep(1)

        except Exception as e:

            print(f"[API ERROR] {e}")

            if attempt == retries:
                return []

            await asyncio.sleep(2)

# =============================================================================
# CONVERT VENUE IDS -> MATRIX INDICES
# =============================================================================

def venue_ids_to_indices(venue_ids, venue_to_idx):

    indices = []

    for vid in venue_ids:

        idx = venue_to_idx.get(vid)

        if idx is not None:
            indices.append(idx)

    return np.array(indices, dtype=int)

# =============================================================================
# MAIN EVALUATION LOOP
# =============================================================================

async def run_llm_evaluation(
    groups,
    df,
    item_id_map,
    itinerary_size=5,
    concurrency=5,
    candidate_pool_size=100
):

    # Reverse map
    venue_to_idx = {
        v: int(k)
        for k, v in item_id_map.items()
    }

    all_venue_ids = list(item_id_map.values())

    sem = asyncio.Semaphore(concurrency)

    async def evaluate_group(group):

        async with sem:

            # Random candidate subset
            candidate_venues = np.random.choice(
                all_venue_ids,
                size=min(candidate_pool_size, len(all_venue_ids)),
                replace=False
            ).tolist()

            venue_ids = await llm_recommend(
                group,
                df,
                candidate_venues,
                itinerary_size
            )

            top_indices = venue_ids_to_indices(
                venue_ids,
                venue_to_idx
            )

            if len(top_indices) == 0:
                return None

            # =========================================================
            # YOUR EXISTING METRIC FUNCTIONS
            # =========================================================

            nsw = compute_nsw(group, top_indices)
            auc = compute_auc(group, top_indices)
            mrr = compute_mrr(group, top_indices)

            return nsw, auc, mrr

    results = await tqdm_asyncio.gather(
        *[evaluate_group(g) for g in groups],
        desc="LLM evaluation"
    )

    # Remove failed groups
    results = [r for r in results if r is not None]

    if len(results) == 0:
        raise ValueError("No successful evaluations.")

    nsw_scores, auc_scores, mrr_scores = zip(*results)

    print("\n==================================================")
    print("LLM ZERO-SHOT RESULTS")
    print("==================================================")

    print(f"NSW : {np.mean(nsw_scores):.4f}")
    print(f"AUC : {np.mean(auc_scores):.4f}")
    print(f"MRR : {np.mean(mrr_scores):.4f}")

    return nsw_scores, auc_scores, mrr_scores

# =============================================================================
# RUN
# =============================================================================

# Example:
#
nsw_scores, auc_scores, mrr_scores = await run_llm_evaluation(
     groups=groups,
     df=df,
     item_id_map=item_id_map,
     itinerary_size=5,
     concurrency=5,
     candidate_pool_size=100
 )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 100 groups
Loaded 578777 reviews


LLM evaluation: 100%|██████████| 100/100 [01:42<00:00,  1.03s/it]


LLM ZERO-SHOT RESULTS
NSW : 0.0062
AUC : 0.0326
MRR : 0.1209


## Adjsutment after improve prompt and data processing

Choose to keep zero shot prompt instead of few shot prompt ;
simpler prompt is probably BETTER, because there is no semantic metadata to reason over.
Instead of long reasoning instructions.
rather than content-based recommendation.

Avoiding metadata/categories helps prevent:

leakage
semantic shortcuts
popularity bias from rich descriptions
overreliance on textual cues
unfair comparison against matrix-based recommenders

## Model- GPT 4O mini

In [ ]:
import pandas as pd
import numpy as np
import json
import asyncio

from openai import AsyncOpenAI
from tqdm.asyncio import tqdm_asyncio
from google.colab import drive
from google.colab import userdata

# =============================================================================
# GOOGLE DRIVE
# =============================================================================

drive.mount('/content/drive')

GoogleDrivePath = '/content/drive/My Drive/FairTrip_Project/'
input_file = GoogleDrivePath + 'group_recommendation_dataset.json'

# =============================================================================
# LOAD DATA
# =============================================================================
with open(input_file, "r") as f:
    loaded_group_data = json.load(f)

groups = [group_info["members"] for group_info in loaded_group_data]

print(f"Loaded {len(groups)} groups")
print(f"Loaded {len(df)} reviews") # This `df` refers to the global df from previous cells.

# =============================================================================
# OPENAI CLIENT
# =============================================================================

openai_api_key = userdata.get('secret-key')

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab secrets.")

client = AsyncOpenAI(api_key=openai_api_key)

# =============================================================================
# PROMPT BUILDER
# =============================================================================

def build_prompt(group, df, candidate_venues, itinerary_size=5):

    member_profiles = []

    for user_id in group:

         # Use only highly-rated venues
        user_reviews = (
            df[
                (df["user_id"] == user_id) &
                (df["stars"] >= 4)
            ][["business_id", "stars"]]
            .sort_values("stars", ascending=False)
            .head(20)
            .to_dict(orient="records")
        )

        member_profiles.append({
            "user_id": user_id,
            "review_history": user_reviews
        })

    prompt = f"""
You are a fairness-aware group recommender system.

Your task is to recommend EXACTLY {itinerary_size} venue IDs for a group.

Recommendation goals:
- maximize overall group satisfaction
- balance preferences fairly across members
- prioritize venues likely preferred by multiple users
- avoid extremely incompatible recommendations

==================================================
GROUP MEMBER HISTORIES
==================================================

{json.dumps(member_profiles, indent=2)}

==================================================
CANDIDATE VENUES
==================================================

You MUST choose ONLY from the following venue IDs:

{json.dumps(candidate_venues, indent=2)}

==================================================
RULES
==================================================

- Recommend EXACTLY {itinerary_size} venue IDs
- Use ONLY candidate venue IDs
- Do NOT invent venue IDs
- Focus primarily on highly-rated venues
- Return ONLY raw JSON
- Do NOT use markdown
- Do NOT include additional text

Required JSON format:

{{
  "recommended_venue_ids": [
    "id1",
    "id2",
    "id3",
    "id4",
    "id5"
  ],
  "reasoning": "brief explanation"
}}
"""

    return prompt

# =============================================================================
# SINGLE GROUP RECOMMENDATION
# =============================================================================

async def llm_recommend(
    group,
    df,
    candidate_venues,
    venue_to_idx,
    itinerary_size=5,
    retries=2
):

    prompt = build_prompt(
        group,
        df,
        candidate_venues,
        itinerary_size
    )

    raw = ""

    for attempt in range(retries + 1):

        try:

            response = await client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert fairness-aware group "
                            "recommendation system. "
                            "You must strictly follow instructions "
                            "and output only valid JSON."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                response_format={"type": "json_object"},
                temperature=0,
                max_completion_tokens=256
            )

            raw = response.choices[0].message.content.strip()

            parsed = json.loads(raw)

            recommended = parsed["recommended_venue_ids"]

            # Keep only valid candidate IDs
            recommended = [
            vid for vid in recommended
            if vid in venue_to_idx              # ← any real venue in the dataset is fine
            ]

            # Remove duplicates
            recommended = list(dict.fromkeys(recommended))

            # Enforce exact size
            recommended = recommended[:itinerary_size]

            # Retry if not enough recommendations
            if len(recommended) < itinerary_size:
                raise ValueError(
                    f"Only received {len(recommended)} valid recommendations."
                )

            return recommended

        except (json.JSONDecodeError, KeyError, ValueError) as e:

            print(f"[Retry {attempt+1}] Parse/Validation Error: {e}")

            if attempt == retries:
                print("\n[FAILED GROUP]")
                print(group)
                print(raw)
                return []

            await asyncio.sleep(1)

        except Exception as e:

            print(f"[API ERROR] {e}")

            if attempt == retries:
                return []

            await asyncio.sleep(2)

# =============================================================================
# CONVERT VENUE IDS -> MATRIX INDICES
# =============================================================================

def venue_ids_to_indices(venue_ids, venue_to_idx):

    indices = []

    for vid in venue_ids:

        idx = venue_to_idx.get(vid)

        if idx is not None:
            indices.append(idx)

    return np.array(indices, dtype=int)

# =============================================================================
# MAIN EVALUATION LOOP
# =============================================================================

async def run_llm_evaluation(
    groups,
    df,
    item_id_map,
    itinerary_size=5,
    concurrency=5,
    candidate_pool_size=100
):

    # Reverse mapping
    venue_to_idx = {
        v: int(k)
        for k, v in item_id_map.items()
    }

    all_venue_ids = list(item_id_map.values())

    sem = asyncio.Semaphore(concurrency)

    async def evaluate_group(group):



        async with sem:

            # Random candidate subset
            candidate_venues = np.random.choice(
                all_venue_ids,
                size=min(candidate_pool_size, len(all_venue_ids)),
                replace=False
            ).tolist()

            venue_ids = await llm_recommend(
                group,
                df,
                candidate_venues,
                venue_to_idx,
                itinerary_size
            )

            if len(venue_ids) == 0:
                return None

            top_indices = venue_ids_to_indices(
                venue_ids,
                venue_to_idx
            )

            if len(top_indices) == 0:
                return None

            # =========================================================
            # METRICS
            # =========================================================

            nsw = compute_nsw(group, top_indices)
            auc = compute_auc(group, top_indices)
            mrr = compute_mrr(group, top_indices)

            return nsw, auc, mrr

    results = await tqdm_asyncio.gather(
        *[evaluate_group(g) for g in groups],
        desc="LLM evaluation"
    )

    # Remove failed groups
    results = [r for r in results if r is not None]

    if len(results) == 0:
        raise ValueError("No successful evaluations.")

    nsw_scores, auc_scores, mrr_scores = zip(*results)

    print("\n==================================================")
    print("LLM ZERO-SHOT RESULTS")
    print("==================================================")

    print(f"NSW : {np.mean(nsw_scores):.4f}")
    print(f"AUC : {np.mean(auc_scores):.4f}")
    print(f"MRR : {np.mean(mrr_scores):.4f}")

    return nsw_scores, auc_scores, mrr_scores

# =============================================================================
# RUN
# =============================================================================

nsw_scores, auc_scores, mrr_scores = await run_llm_evaluation(
    groups=groups,
    df=df,
    item_id_map=item_id_map,
    itinerary_size=5,
    concurrency=5,
    candidate_pool_size=100
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 100 groups
Loaded 578777 reviews


LLM evaluation:  10%|█         | 10/100 [00:12<01:41,  1.13s/it]

[Retry 1] Parse/Validation Error: Only received 4 valid recommendations.


LLM evaluation:  14%|█▍        | 14/100 [00:16<01:21,  1.05it/s]

[Retry 2] Parse/Validation Error: Only received 4 valid recommendations.


...

[Retry 2] Parse/Validation Error: Only received 4 valid recommendations.


LLM evaluation:  24%|██▍       | 24/100 [00:28<01:29,  1.17s/it]

[Retry 3] Parse/Validation Error: Only received 4 valid recommendations.

[FAILED GROUP]
['itbJ_tBiBvLpCAtfpU4SOw', '2UPS0rLvGREkxgFcSaje6Q', '2HGxY9bhMmXFVYtk-rqSbw', '9UeHVTMLdCoxlzxXdaRxqg', 'NWFqEfs5Z9wUmCeVfTTDAA']
{
  "recommended_venue_ids": [
    "Ep_jh1Pt4Ggyla21f-BQcQ",
    "LO5yBY9uoowtq7FsaKPJXA",
    "vUrTGX_7HxqeoQ_6QCVz6g",
    "N30ggGzFpXvc2NZYwOW3qg",
    "h85b3MFPWyaopMBy8DpBpA"
  ],
  "reasoning": "These venues have high ratings from multiple group members, ensuring a balance of preferences and maximizing overall satisfaction."
}


LLM evaluation: 100%|██████████| 100/100 [01:37<00:00,  1.03it/s]


LLM ZERO-SHOT RESULTS
NSW : 0.0176
AUC : 0.0775
MRR : 0.1697



## Improvement Analytics :
*   The biggest driver is almost certainly the stars >= 4 filter — giving the LLM clean, preference-positive signal — combined with the stricter validation ensuring only genuinely valid recommendations reach the metrics.
*   including system prompt also likely to improve the result by giving llm a better understanding of the task.


*    More Explicit Prompt Instructions such as :

     "prioritize venues likely preferred by multiple users"

     "avoid extremely incompatible recommendations"

     "Focus primarily on highly-rated venues"

     These steer the model toward fairness-aware behavior more explicitly.




## Model - gpt-5.4-mini

In [ ]:
import pandas as pd
import numpy as np
import json
import asyncio

from openai import AsyncOpenAI
from tqdm.asyncio import tqdm_asyncio
from google.colab import drive
from google.colab import userdata

# =============================================================================
# GOOGLE DRIVE
# =============================================================================

drive.mount('/content/drive')

GoogleDrivePath = '/content/drive/My Drive/FairTrip_Project/'
input_file = GoogleDrivePath + 'group_recommendation_dataset.json'

# =============================================================================
# LOAD DATA
# =============================================================================
with open(input_file, "r") as f:
    loaded_group_data = json.load(f)

groups = [group_info["members"] for group_info in loaded_group_data]

print(f"Loaded {len(groups)} groups")
print(f"Loaded {len(df)} reviews") # This `df` refers to the global df from previous cells.

# =============================================================================
# OPENAI CLIENT
# =============================================================================

openai_api_key = userdata.get('secret-key')

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in Colab secrets.")

client = AsyncOpenAI(api_key=openai_api_key)

# =============================================================================
# PROMPT BUILDER
# =============================================================================

def build_prompt(group, df, candidate_venues, itinerary_size=5):

    member_profiles = []

    for user_id in group:

         # Use only highly-rated venues
        user_reviews = (
            df[
                (df["user_id"] == user_id) &
                (df["stars"] >= 4)
            ][["business_id", "stars"]]
            .sort_values("stars", ascending=False)
            .head(20)
            .to_dict(orient="records")
        )

        member_profiles.append({
            "user_id": user_id,
            "review_history": user_reviews
        })

    prompt = f"""
You are a fairness-aware group recommender system.

Your task is to recommend EXACTLY {itinerary_size} venue IDs for a group.

Recommendation goals:
- maximize overall group satisfaction
- balance preferences fairly across members
- prioritize venues likely preferred by multiple users
- avoid extremely incompatible recommendations

==================================================
GROUP MEMBER HISTORIES
==================================================

{json.dumps(member_profiles, indent=2)}

==================================================
CANDIDATE VENUES
==================================================

You MUST choose ONLY from the following venue IDs:

{json.dumps(candidate_venues, indent=2)}

==================================================
RULES
==================================================

- Recommend EXACTLY {itinerary_size} venue IDs
- Use ONLY candidate venue IDs
- Do NOT invent venue IDs
- Focus primarily on highly-rated venues
- Return ONLY raw JSON
- Do NOT use markdown
- Do NOT include additional text

Required JSON format:

{{
  "recommended_venue_ids": [
    "id1",
    "id2",
    "id3",
    "id4",
    "id5"
  ],
  "reasoning": "brief explanation"
}}
"""

    return prompt

# =============================================================================
# SINGLE GROUP RECOMMENDATION
# =============================================================================

async def llm_recommend(
    group,
    df,
    candidate_venues,
    venue_to_idx,
    itinerary_size=5,
    retries=2
):

    prompt = build_prompt(
        group,
        df,
        candidate_venues,
        itinerary_size
    )

    raw = ""

    for attempt in range(retries + 1):

        try:

            response = await client.chat.completions.create(
                model="gpt-5.4-mini",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert fairness-aware group "
                            "recommendation system. "
                            "You must strictly follow instructions "
                            "and output only valid JSON."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                response_format={"type": "json_object"},
                temperature=0,
                max_completion_tokens=256
            )

            raw = response.choices[0].message.content.strip()

            parsed = json.loads(raw)

            recommended = parsed["recommended_venue_ids"]

            # Keep only valid candidate IDs
            recommended = [
            vid for vid in recommended
            if vid in venue_to_idx              # ← any real venue in the dataset is fine
            ]

            # Remove duplicates
            recommended = list(dict.fromkeys(recommended))

            # Enforce exact size
            recommended = recommended[:itinerary_size]

            # Retry if not enough recommendations
            if len(recommended) < itinerary_size:
                raise ValueError(
                    f"Only received {len(recommended)} valid recommendations."
                )

            return recommended

        except (json.JSONDecodeError, KeyError, ValueError) as e:

            print(f"[Retry {attempt+1}] Parse/Validation Error: {e}")

            if attempt == retries:
                print("\n[FAILED GROUP]")
                print(group)
                print(raw)
                return []

            await asyncio.sleep(1)

        except Exception as e:

            print(f"[API ERROR] {e}")

            if attempt == retries:
                return []

            await asyncio.sleep(2)

# =============================================================================
# CONVERT VENUE IDS -> MATRIX INDICES
# =============================================================================

def venue_ids_to_indices(venue_ids, venue_to_idx):

    indices = []

    for vid in venue_ids:

        idx = venue_to_idx.get(vid)

        if idx is not None:
            indices.append(idx)

    return np.array(indices, dtype=int)

# =============================================================================
# MAIN EVALUATION LOOP
# =============================================================================

async def run_llm_evaluation(
    groups,
    df,
    item_id_map,
    itinerary_size=5,
    concurrency=5,
    candidate_pool_size=100
):

    # Reverse mapping
    venue_to_idx = {
        v: int(k)
        for k, v in item_id_map.items()
    }

    all_venue_ids = list(item_id_map.values())

    sem = asyncio.Semaphore(concurrency)

    async def evaluate_group(group):



        async with sem:

            # Random candidate subset
            candidate_venues = np.random.choice(
                all_venue_ids,
                size=min(candidate_pool_size, len(all_venue_ids)),
                replace=False
            ).tolist()

            venue_ids = await llm_recommend(
                group,
                df,
                candidate_venues,
                venue_to_idx,
                itinerary_size
            )

            if len(venue_ids) == 0:
                return None

            top_indices = venue_ids_to_indices(
                venue_ids,
                venue_to_idx
            )

            if len(top_indices) == 0:
                return None

            # =========================================================
            # METRICS
            # =========================================================

            nsw = compute_nsw(group, top_indices)
            auc = compute_auc(group, top_indices)
            mrr = compute_mrr(group, top_indices)

            return nsw, auc, mrr

    results = await tqdm_asyncio.gather(
        *[evaluate_group(g) for g in groups],
        desc="LLM evaluation"
    )

    # Remove failed groups
    results = [r for r in results if r is not None]

    if len(results) == 0:
        raise ValueError("No successful evaluations.")

    nsw_scores, auc_scores, mrr_scores = zip(*results)

    print("\n==================================================")
    print("LLM ZERO-SHOT RESULTS")
    print("==================================================")

    print(f"NSW : {np.mean(nsw_scores):.4f}")
    print(f"AUC : {np.mean(auc_scores):.4f}")
    print(f"MRR : {np.mean(mrr_scores):.4f}")

    return nsw_scores, auc_scores, mrr_scores

# =============================================================================
# RUN
# =============================================================================

nsw_scores, auc_scores, mrr_scores = await run_llm_evaluation(
    groups=groups,
    df=df,
    item_id_map=item_id_map,
    itinerary_size=5,
    concurrency=5,
    candidate_pool_size=100
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 100 groups
Loaded 578777 reviews


LLM evaluation:  96%|█████████▌| 96/100 [01:15<00:03,  1.32it/s]

[Retry 1] Parse/Validation Error: Only received 4 valid recommendations.


LLM evaluation:  99%|█████████▉| 99/100 [01:17<00:00,  1.37it/s]

[Retry 2] Parse/Validation Error: Only received 4 valid recommendations.


LLM evaluation: 100%|██████████| 100/100 [01:21<00:00,  1.23it/s]


LLM ZERO-SHOT RESULTS
NSW : 0.0067
AUC : 0.0203
MRR : 0.1262


## Why GPT-4o Can Outperform GPT-5.4 on This Task

##  GPT 5.4 mini consider as an agentic wrkflow for more complex and multi steps reasoning task, however our instruction is simple and deterministic; 5.4 mini can overthink. In conclusion, more capatble doesn't mean good for every task.